In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import random

# -------------------------------------------------------------------
# Load required data
# -------------------------------------------------------------------
RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")

# Category tree
cat_tree = pd.read_csv(RAW_DIR / "category_tree.csv")
parent_of = {}
children_of = {}
for _, row in cat_tree.iterrows():
    child = int(row["categoryid"])
    if pd.notna(row["parentid"]):
        parent = int(row["parentid"])
        parent_of[child] = parent
        children_of.setdefault(parent, []).append(child)
roots = cat_tree[cat_tree["parentid"].isna()]["categoryid"].tolist()

# Session features and clusters
session_df = pd.read_parquet(PROC_DIR / "user_sessions.parquet")
cluster_df = pd.read_csv(PROC_DIR / "user_clusters.csv")
session_df = session_df.merge(cluster_df, on=['visitorid', 'session_id'])

# Events for item history
events = pd.read_parquet(PROC_DIR / "events_with_sessions.parquet")

# Top items for synthetic KNN history
item_counts = events['itemid'].value_counts()
top_items = item_counts.head(5000).index.tolist()

# Helper: get root of a node
def get_root(node):
    while node in parent_of:
        node = parent_of[node]
    return node

# Helper: get all descendants of a root
def get_descendants(root):
    desc = set()
    stack = children_of.get(root, [])
    while stack:
        node = stack.pop()
        if node in desc:
            continue
        desc.add(node)
        stack.extend(children_of.get(node, []))
    return desc

# -------------------------------------------------------------------
# 1. A* Search: valid (root, goal) pairs
# -------------------------------------------------------------------
astar_pairs = []
for root in roots[:10]:
    desc = get_descendants(root)
    if desc:
        for goal in random.sample(list(desc), min(3, len(desc))):
            astar_pairs.append({"Start (root)": root, "Goal": goal})
if len(astar_pairs) < 10:
    for root in roots[10:]:
        desc = get_descendants(root)
        if desc:
            for goal in random.sample(list(desc), min(2, len(desc))):
                astar_pairs.append({"Start (root)": root, "Goal": goal})
astar_pairs = astar_pairs[:15]

# -------------------------------------------------------------------
# 2. Purchase Prediction: synthetic feature vectors
# -------------------------------------------------------------------
prediction_examples = []
for _ in range(10):
    row = {
        "num_views": np.random.randint(1, 20),
        "num_addtocart": np.random.randint(0, 5),
        "unique_items": np.random.randint(1, 15),
        "categories_viewed": np.random.randint(1, 10),
        "duration_min": round(np.random.uniform(1, 60), 1),
        "cluster": np.random.randint(0, 4)
    }
    prediction_examples.append(row)

# -------------------------------------------------------------------
# 3. Contextual Bandit - Existing Session IDs (with item history)
# -------------------------------------------------------------------
sessions_with_items = events.groupby(['visitorid', 'session_id']).size().reset_index(name='count')
sessions_with_items = sessions_with_items[sessions_with_items['count'] > 0]
existing_sessions = sessions_with_items.head(15).to_dict('records')[:10]
existing_examples = [{"visitorid": s['visitorid'], "session_id": s['session_id']} for s in existing_sessions]

# -------------------------------------------------------------------
# 4. Contextual Bandit - New Session (synthetic)
# -------------------------------------------------------------------
new_session_examples = []
for _ in range(10):
    features = {
        "num_views": np.random.randint(1, 20),
        "num_addtocart": np.random.randint(0, 5),
        "unique_items": np.random.randint(1, 15),
        "categories_viewed": np.random.randint(1, 10),
        "duration_min": round(np.random.uniform(1, 60), 1),
        "cluster": np.random.randint(0, 4),
        "item_history (for KNN)": random.sample(top_items, min(3, len(top_items)))
    }
    new_session_examples.append(features)

# -------------------------------------------------------------------
# Print all examples using pandas (no external library)
# -------------------------------------------------------------------
def print_df(df, title):
    print(title)
    print(df.to_string(index=False))
    print("\n")

print_df(pd.DataFrame(astar_pairs), "1. A* SEARCH: Valid (Start Root, Goal) Pairs")
print_df(pd.DataFrame(prediction_examples), "2. PURCHASE PREDICTION: Sample Input Values")
print_df(pd.DataFrame(existing_examples), "3. CONTEXTUAL BANDIT: Existing Session IDs (with item history)")
# For new sessions, convert list to string for display
df_new = pd.DataFrame(new_session_examples)
df_new['item_history (for KNN)'] = df_new['item_history (for KNN)'].apply(lambda x: ', '.join(map(str, x)))
print_df(df_new, "4. CONTEXTUAL BANDIT: New Session (synthetic)")

# -------------------------------------------------------------------
# Save to CSV
# -------------------------------------------------------------------
pd.DataFrame(astar_pairs).to_csv("../outputs/astar_test_examples.csv", index=False)
pd.DataFrame(prediction_examples).to_csv("../outputs/prediction_test_examples.csv", index=False)
pd.DataFrame(existing_examples).to_csv("../outputs/bandit_existing_sessions.csv", index=False)
df_new.to_csv("../outputs/bandit_new_sessions.csv", index=False)
print("\nAll test examples saved to 'outputs/' folder.")

1. A* SEARCH: Valid (Start Root, Goal) Pairs
 Start (root)  Goal
          791  1362
          791  1423
          791   909
         1490  1282
         1490   407
         1490  1343
          431   447
          431   952
          431   301
          755   864
          755  1449
          755   190
          378  1535
          378  1580
          378   894


2. PURCHASE PREDICTION: Sample Input Values
 num_views  num_addtocart  unique_items  categories_viewed  duration_min  cluster
         2              4            11                  6          43.4        1
        11              1            10                  8          11.0        3
         7              4             9                  6          49.6        0
        16              4             8                  9          21.5        2
        16              3            12                  3           7.6        0
        11              3             3                  1          12.8        2
         7     